# Week 1: Torque vs. preload with pandas

**Goal:** learn the core pandas workflow (build a table, explore it, group it, plot it, fit a model) on data shaped like your thesis tests.

The data is simulated with the short-form torque equation **T = K · F · d**. K stays at 0.24 up to 90 N·m, then rises at 100 N·m, the same behavior as your thesis. Later you'll swap in your real CSV.

**How to use:** run each cell with **Shift + Enter**, top to bottom. Read the code before you run it.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("pandas", pd.__version__, "| numpy", np.__version__)

## 2. Simulate a tightening test

8 torque steps (30–100 N·m), 5 repeats each, with load-cell noise. Each test becomes one row in a **DataFrame**, pandas' version of a spreadsheet.

In [ ]:
rng = np.random.default_rng(seed=0)

d = 0.012                              # M12 bolt diameter [m]
torque_steps = np.arange(30, 110, 10)  # 30, 40, ..., 100 N*m
repeats = 5
noise_N = 300                          # load cell noise (std) [N]

def nut_factor(T):
    """K is steady at 0.24, then rises above 90 N*m as gasket stress drops."""
    return 0.24 if T <= 90 else 0.24 + 0.007 * (T - 90)

rows = []
for T in torque_steps:
    for r in range(1, repeats + 1):
        F = T / (nut_factor(T) * d)
        rows.append({"torque_Nm": T, "repeat": r, "preload_N": F + rng.normal(0, noise_N)})

df = pd.DataFrame(rows)
df.head(10)

## 3. Look at the data

Always do this first: shape, column types, missing values, basic statistics.

In [ ]:
df.info()

In [ ]:
df.describe().round(1)

## 4. Group by torque step

`groupby` + `agg` is the most used pandas pattern: split the rows into groups, summarize each one.

In [ ]:
summary = df.groupby("torque_Nm")["preload_N"].agg(["mean", "std", "count"])
summary.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(df["torque_Nm"], df["preload_N"] / 1000, alpha=0.4, label="each test")
ax.errorbar(summary.index, summary["mean"] / 1000, yerr=summary["std"] / 1000,
            fmt="o-", color="black", capsize=4, label="mean ± std")
ax.set_xlabel("Tightening torque [N·m]")
ax.set_ylabel("Clamping force [kN]")
ax.set_title("Torque vs. preload")
ax.legend()
plt.show()

## 5. Nut factor for every test

Adding a column computed from other columns is one line in pandas.

In [ ]:
df["K"] = df["torque_Nm"] / (df["preload_N"] * d)
K_summary = df.groupby("torque_Nm")["K"].agg(["mean", "std"])
K_summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.lineplot(data=df, x="torque_Nm", y="K", marker="o", errorbar="sd", ax=ax)
ax.axhline(0.24, color="gray", linestyle="--", label="K = 0.24")
ax.set_xlabel("Tightening torque [N·m]")
ax.set_ylabel("Nut factor K [-]")
ax.set_title("Nut factor vs. torque")
ax.legend()
plt.show()

## 6. Fit K on the linear region only

Filtering rows with a condition (`df[df[...] <= 90]`) is the other pattern you'll use every day.

In [ ]:
linear = df[df["torque_Nm"] <= 90]
slope = np.sum(linear["torque_Nm"] * linear["preload_N"]) / np.sum(linear["torque_Nm"] ** 2)
K_fit = 1 / (slope * d)
print(f"Fitted K on 30–90 N·m: {K_fit:.3f}  (true value 0.240)")

## 7. Your turn

Write the code yourself first. Ask for help only after you've tried.

1. Change `noise_N` in section 2 from `300` to `1500`, then **Runtime → Run all**. What happens to the `std` column and to `K_fit`?
2. Use pandas to find the first torque step where the mean K is above 0.26. *Hint:* filter `K_summary`.
3. Add a column `preload_kN` with the preload in kN.
4. Save `df` to `torque_tests.csv` with `df.to_csv(...)`, then find it in the 📁 Files panel on the left.

In [ ]:
# Exercise 2


In [ ]:
# Exercise 3


In [ ]:
# Exercise 4


## 8. Load your real thesis data

Colab deletes uploaded files when the session ends. Keep data in **Google Drive** so it stays (option B).

Uncomment one option (select the lines and press **Ctrl + /**), then run it.

In [ ]:
# Option A: upload from your PC (temporary)
# from google.colab import files
# uploaded = files.upload()
# real = pd.read_csv(next(iter(uploaded)))

# Option B: read from Google Drive (permanent)
# from google.colab import drive
# drive.mount("/content/drive")
# real = pd.read_csv("/content/drive/MyDrive/ml-journey/data/your_file.csv")

# real.head()